# Principal component analysis with real wheat-kernel data

This notebook reproduces the worked example in the Praxagent PCA Deep Dive. It downloads the open **UCI Seeds** dataset, checks the source file, scales seven kernel measurements, fits principal component analysis (PCA), and creates the score and loading plots.

No programming experience is assumed. Run the cells from top to bottom.

## 1. Load the Python tools

`pandas` handles tables, `matplotlib` draws figures, and scikit-learn supplies the scaler and PCA implementation. These packages are already available in Google Colab.

In [ ]:
from io import BytesIO
import hashlib
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

## 2. Download and verify the open dataset

The UCI file contains 210 rows. Each row is one wheat kernel. The first seven columns are measurements derived from soft X-ray images. The last column is the recorded variety code: 1 is Kama, 2 is Rosa, and 3 is Canadian.

This is a **pre-cleaned** public table: all seven measurement columns are numeric, there are no blank or `NaN` measurement cells, the variety codes are consistent, and one row already represents one kernel. Raw laboratory exports often require additional work to identify missing-value markers, typing mistakes, impossible values, mixed units, duplicates, and technical repeats. Pre-cleaned does not mean guaranteed error-free, so we still inspect the table before PCA.

The checksum protects this tutorial from silently analyzing a different file if the download changes.

In [ ]:
DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00236/"
    "seeds_dataset.txt"
)
EXPECTED_SHA256 = "1f3f83c0d8485ae9148061389d19628607e3f5660e3d6f40ec9102fb398bb12f"

raw_bytes = urlopen(DATA_URL, timeout=30).read()
observed_sha256 = hashlib.sha256(raw_bytes).hexdigest()
assert observed_sha256 == EXPECTED_SHA256, observed_sha256
print("Verified UCI source file:", observed_sha256)

In [ ]:
FEATURES = [
    "area",
    "perimeter",
    "compactness",
    "kernel_length",
    "kernel_width",
    "asymmetry_coefficient",
    "kernel_groove_length",
]
VARIETIES = {1: "Kama", 2: "Rosa", 3: "Canadian"}

seeds = pd.read_csv(
    BytesIO(raw_bytes),
    sep=r"\s+",
    header=None,
    names=[*FEATURES, "variety_code"],
)
seeds.insert(0, "kernel_id", [f"kernel_{i:03d}" for i in range(1, len(seeds) + 1)])
seeds["variety"] = seeds["variety_code"].map(VARIETIES)
assert seeds["variety"].notna().all()
seeds.head()

**What the mapping line does:** `seeds["variety_code"]` selects the existing numerical codes. `.map(VARIETIES)` looks up each code in `{1: "Kama", 2: "Rosa", 3: "Canadian"}`. `seeds["variety"] =` stores those names in a new column without overwriting the original codes. An unknown code would become a missing value, so the following `assert` stops the notebook if any code was not mapped.

**You should see:** five kernels, seven measurement columns, a variety code, and a variety name. The identifier and variety columns are metadata. They will not be included in the PCA calculation.

## 3. Check what one row means and whether values are missing

This dataset has no repeat identifier, plant identifier, field-plot identifier, or plate identifier. We therefore cannot honestly combine rows into plant-level or plot-level summaries. One row remains one measured kernel.

In [ ]:
print("Table shape:", seeds.shape)
print("Kernels per variety:")
print(seeds["variety"].value_counts())
print("\nMissing values per measurement:")
print(seeds[FEATURES].isna().sum())

assert len(seeds) == 210
assert seeds[FEATURES].isna().sum().sum() == 0
assert seeds["variety"].value_counts().to_dict() == {
    "Kama": 70, "Rosa": 70, "Canadian": 70
}

**You should see:** 210 rows, 70 kernels in each variety, and zero missing measurement cells. Because nothing is missing, this analysis does not impute, or fill in, any values.

## 4. Look at two original measurements

A length-versus-width plot is useful, but it can show only two of the seven measurement directions. PCA will use all seven together. The variety labels are added only for interpretation; PCA does not receive them.

In [ ]:
COLORS = {"Kama": "#4B6787", "Rosa": "#A67C52", "Canadian": "#6F8D5E"}
MARKERS = {"Kama": "o", "Rosa": "s", "Canadian": "^"}

fig, ax = plt.subplots(figsize=(9, 5))
for variety in ["Kama", "Rosa", "Canadian"]:
    rows = seeds["variety"] == variety
    ax.scatter(
        seeds.loc[rows, "kernel_length"],
        seeds.loc[rows, "kernel_width"],
        label=variety, color=COLORS[variety], marker=MARKERS[variety], alpha=0.8
    )
ax.set(xlabel="Kernel length", ylabel="Kernel width",
       title="Two of the seven original measurements")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

### 4a. Show all seven original measurements

The two-axis chart hides five columns. These seven small panels keep every feature on its own original scale. Every symbol is one measured kernel. Each box marks the middle half of a variety's values, with the median shown inside.

In [ ]:
rng = np.random.default_rng(236)
fig, axes = plt.subplots(4, 2, figsize=(12, 12))

for feature, ax in zip(FEATURES, axes.flat):
    groups = [seeds.loc[seeds["variety"] == name, feature] for name in VARIETIES.values()]
    ax.boxplot(groups, positions=[0, 1, 2], vert=False, widths=0.5, showfliers=False)
    for y, variety in enumerate(VARIETIES.values()):
        values = seeds.loc[seeds["variety"] == variety, feature]
        jitter = rng.uniform(-0.16, 0.16, len(values))
        ax.scatter(values, y + jitter, color=COLORS[variety],
                   marker=MARKERS[variety], s=18, alpha=0.65)
    readable_name = feature.replace("_", " ")
    ax.set(yticks=[0, 1, 2], yticklabels=list(VARIETIES.values()),
           xlabel=readable_name, title=readable_name.title())
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.25)

axes.flat[-1].remove()
plt.tight_layout()
plt.show()

### 4b. See which measurements move together

A Pearson correlation near +1 means two measurements rise together in an almost straight-line pattern. A value near -1 means one tends to fall as the other rises. A value near 0 means little straight-line relationship. Correlation does not establish causation.

For paired measurements $x_i$ and $y_i$ on kernel $i$, the calculation is

$$r_{xy}=\frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i-\bar{x})^2}\sqrt{\sum_{i=1}^{n}(y_i-\bar{y})^2}}.$$

Subtracting each feature's average records whether every kernel is above or below that average. Multiplying the paired deviations gives a positive value when both move in the same direction and a negative value when they move in opposite directions. Adding those products reveals the overall direction. The denominator removes the original units and constrains the answer to the range -1 through +1.

In [ ]:
corr = seeds[FEATURES].corr()
fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
short_names = [name.replace("kernel_", "").replace("_coefficient", "").replace("_", " ")
               for name in FEATURES]
ax.set_xticks(range(7), short_names, rotation=40, ha="right")
ax.set_yticks(range(7), short_names)
for row in range(7):
    for column in range(7):
        ax.text(column, row, f"{corr.iloc[row, column]:.2f}",
                ha="center", va="center")
fig.colorbar(image, ax=ax, label="Pearson correlation")
ax.set_title("How the original measurements move together")
plt.tight_layout()
plt.show()

print("Area with perimeter:", round(corr.loc["area", "perimeter"], 2))
print("Asymmetry with groove length:",
      round(corr.loc["asymmetry_coefficient", "kernel_groove_length"], 2))

**You should see:** area and perimeter correlate at 0.99, while asymmetry and groove length correlate at -0.01. Several size-related columns therefore carry overlapping information, which gives us a concrete reason to try dimensionality reduction.

## 5. Scale the seven measurements

Compactness is numerically close to 1, while area is commonly above 10 in this table. PCA is sensitive to numerical scale.

Before scaling, remember that the mean locates the center of one feature column, while the standard deviation summarizes how widely its values spread around that center. A small standard deviation means values tend to stay close to the mean; a larger one means they tend to sit farther away. `StandardScaler` uses the population standard deviation, which squares each value's difference from the mean, averages those squared differences using `n`, and takes the square root.

`StandardScaler` then works on one feature column at a time: it subtracts that feature's mean from every value, then divides by that feature's standard deviation.

In ordinary words: `standardized value = (observed value - feature average) / feature standard deviation`. Subtraction tells us whether a kernel is above or below the feature average. Division tells us how large that difference is compared with the feature's usual spread.

For example, kernel 1 has kernel length 5.763. The full column has mean 5.6285 and population standard deviation 0.4420, so its standardized length is `(5.763 - 5.6285) / 0.4420`, or about `+0.30`. That means 0.30 standard deviations above the mean length. A standardized value of 0 is at the mean, +1 is one standard deviation above it, and -1 is one standard deviation below it. Scaling does not make a distribution bell-shaped or remove outliers.

In [ ]:
X = seeds[FEATURES]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_check = pd.DataFrame(X_scaled, columns=FEATURES).agg(["mean", "std"])
scaled_check.round(3)

The scaled means should round to 0. The displayed sample standard deviations round to about 1.002 because pandas uses a different denominator for this summary than `StandardScaler`; that tiny difference is expected.

## 6. Fit PCA and give every kernel its new coordinates

`fit_transform` learns seven new component axes and returns seven coordinates, or scores, for every kernel. We orient the component signs to match the article. A mirrored component would describe the same PCA geometry.

In [ ]:
pca = PCA(svd_solver="full")
scores = pca.fit_transform(X_scaled)

# Component signs are arbitrary. These two checks keep our printed signs stable.
if pca.components_[0, FEATURES.index("area")] < 0:
    pca.components_[0] *= -1
    scores[:, 0] *= -1
if pca.components_[1, FEATURES.index("compactness")] < 0:
    pca.components_[1] *= -1
    scores[:, 1] *= -1

variance_percent = pca.explained_variance_ratio_ * 100
print("Variance represented by PC1:", round(variance_percent[0], 1), "%")
print("Variance represented by PC2:", round(variance_percent[1], 1), "%")
print("PC1 plus PC2:", round(variance_percent[:2].sum(), 1), "%")

assert np.isclose(pca.explained_variance_ratio_[0], 0.718743027)
assert np.isclose(pca.explained_variance_ratio_[1], 0.171081835)

**You should see:** PC1 represents 71.9%, PC2 represents 17.1%, and together they represent 89.0% of the variation in the scaled seven-feature table.

In [ ]:
score_table = seeds[["kernel_id", "variety_code", "variety"]].copy()
for component in range(scores.shape[1]):
    score_table[f"PC{component + 1}"] = scores[:, component]
score_table.head()

## 7. Plot the PCA scores and explained variance

The left panel shows each kernel's PC1 and PC2 scores. The right panel shows how much variation each component represents. The variety labels did not help PCA choose the axes; they are used only after fitting so we can interpret where known varieties land.

In [ ]:
fig, (ax_scores, ax_variance) = plt.subplots(1, 2, figsize=(13, 5))
for variety in ["Kama", "Rosa", "Canadian"]:
    rows = score_table["variety"] == variety
    ax_scores.scatter(
        score_table.loc[rows, "PC1"], score_table.loc[rows, "PC2"],
        label=variety, color=COLORS[variety], marker=MARKERS[variety], alpha=0.8
    )
ax_scores.axhline(0, color="0.6", linewidth=1)
ax_scores.axvline(0, color="0.6", linewidth=1)
ax_scores.set(
    xlabel=f"PC1 ({variance_percent[0]:.1f}% of variance)",
    ylabel=f"PC2 ({variance_percent[1]:.1f}% of variance)",
    title="Kernel scores",
)
ax_scores.legend()
ax_scores.grid(alpha=0.25)

components = np.arange(1, 8)
ax_variance.bar(components, variance_percent, color=["#4B6787"] * 2 + ["#C9BFB3"] * 5)
ax_variance.set(
    xlabel="Principal component", ylabel="Variance represented (%)",
    title="Explained variance", xticks=components,
)
ax_variance.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Use principal-axis coefficients to translate the axes back into measurements

Scikit-learn calls the rows of `components_` the principal axes. After transposing, each value is a principal-axis coefficient describing how an original feature helps define a component. Many tutorials informally call these values **loadings**, but some statistical texts reserve that word for feature-component correlations. This notebook plots principal-axis coefficients, not correlation loadings. A large absolute coefficient matters more than a value near zero. The sign gives direction, but every component can be mirrored without changing the scientific content.

In [ ]:
component_coefficients = pd.DataFrame(
    pca.components_.T,
    index=FEATURES,
    columns=[f"PC{i}" for i in range(1, 8)],
)
component_coefficients[["PC1", "PC2"]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for component, ax in zip(["PC1", "PC2"], axes):
    values = component_coefficients[component]
    colors = ["#4B6787" if value >= 0 else "#A67C52" for value in values]
    ax.barh(component_coefficients.index, values, color=colors)
    ax.axvline(0, color="0.5", linewidth=1)
    ax.set(xlabel="Principal-axis coefficient", title=component, xlim=(-0.82, 0.82))
    ax.grid(axis="x", alpha=0.25)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

In this fitted orientation, PC1 rises with area, perimeter, kernel length, width, groove length, and compactness, while the asymmetry coefficient points weakly in the other direction. PC2 contrasts compactness with the asymmetry coefficient and groove length. These principal-axis coefficients describe the axes. They do not say that any feature causes a variety difference.

## 9. Save the exact scores and principal-axis coefficients

These files let you inspect the numerical values behind the plots.

In [ ]:
score_table.to_csv("wheat_kernel_pca_scores.csv", index=False)
component_coefficients.to_csv("wheat_kernel_pca_component_coefficients.csv")
print("Saved wheat_kernel_pca_scores.csv and wheat_kernel_pca_component_coefficients.csv")

## 10. If you later build a classifier, split before preprocessing

The full-data scaling above is appropriate for describing this complete table. It must not be reused to claim fair performance on held-out kernels. For predictive evaluation, put every learned step inside a pipeline. The median imputer below changes nothing in this complete dataset, but shows where imputation would belong if future rows contained missing values.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

leakage_safe_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    PCA(n_components=4, svd_solver="full"),
    LogisticRegression(max_iter=2000),
)
leakage_safe_pipeline

This notebook intentionally stops before reporting classifier accuracy. PCA is the lesson, and the source table does not provide field, plate, harvest, or repeated-plant identifiers needed to design a strong biological generalization test.